# 连接网络
**scikit-rf** 支持连接 N 端口网络的任意端口。它使用一种称为子网络增长的算法[[1]](#参考文献)，通过 `connect()` 函数实现。请注意，此函数会考虑端口阻抗。如果两个连接的端口具有不同的端口阻抗，则会插入适当的阻抗失配。此功能在此用常遇到的情况进行说明。

In [ ]:
import skrf as rf

## 级联 2 端口和 1 端口网络
一个常见的问题是将两个网络相互连接，也称为级联网络，这创建一个新的网络。下图说明了简单情况，其中端口号以灰色标识：

<img src="figures/networks_connecting_2_2ports.svg" width="600">

或者，

<img src="figures/networks_connecting_2port_1port.svg" width="600">


让我们通过将传输线（2 端口网络）连接到短路（1 端口网络）来创建延迟短路（1 端口网络）来说明这一点：

<img src="figures/networks_delay_short.svg" width="600">

级联网络是一个频繁的操作，可以通过 `**` 运算符或 `cascade` 函数方便地完成：

In [ ]:
line = rf.data.wr2p2_line  # 2-port
short = rf.data.wr2p2_short  # 1-port

delayshort = line ** short  # --> 1-port Network
print(delayshort)

或者，等效地使用 `cascade()` 函数：

In [ ]:
delayshort2 = rf.network.cascade(line, short)
print(delayshort2 == delayshort)  # the result is the same

当然可以使用 `connect()` 函数连接两个 2 端口网络。`connect()` 函数需要网络和要连接的端口号。在我们的示例中，线的端口 1 连接到短路的端口 0：

In [ ]:
delayshort3 = rf.network.connect(line, 1, short, 0)
print(delayshort3 == delayshort)

通常需要将一系列网络级联在一起：

<img src="figures/networks_connecting_N_2ports.svg" width="700">
或者，
<img src="figures/networks_connecting_N_2ports_1port.svg" width="700">


可以使用链式 `**` 或方便的函数 `cascade_list` 来实现：

In [ ]:
line1 = rf.data.wr2p2_line  # 2-port
line2 = rf.data.wr2p2_line  # 2-port
line3 = rf.data.wr2p2_line  # 2-port
line4 = rf.data.wr2p2_line  # 2-port
short = rf.data.wr2p2_short  # 1-port

chain1 = line1 ** line2 ** line3 ** line4 ** short

chain2 = rf.network.cascade_list([line1, line2, line3, line4, short])

print(chain1 == chain2)

## 级联 2N 端口网络

级联运算符 `**` 也适用于 2N 端口网络，使用以下端口方案：

<img src="figures/networks_connecting_2_2Nports.svg" width="600">

它也适用于多个 2N 端口网络。例如，假设你要级联三个 4 端口网络 `ntw1`、`ntw2` 和 `ntw3`，你可以使用：
```
resulting_ntw = ntw1 ** ntw2 ** ntw3
``` 
这在[关于平衡网络的示例](../examples/networktheory/Balanced%20Network%20De-embedding.ipynb)中有所说明。

## 级联多端口网络
要对多端口网络进行特定连接，有三种解决方案可用，主要取决于要构建的电路的复杂程度：

* 对于少量连接：`connect()` 函数

* 对于中等复杂度，可能需要将多个网络并联连接到同一交叉点，我们提供 `parallelconnect()` 方法。此方法在 `connect()` 的简单性和 `Circuit` 对象的灵活性之间提供了平衡。有关更多信息，请参阅 [`paralleconnect()` 文档](../api/generated/skrf.network.paralleconnect.html#skrf.network.paralleconnect)

* 对于许多任意 N 端口网络之间更高级的连接，`Circuit` 对象更相关，因为它允许明确定义端口和网络之间的连接。有关更多信息，请参阅 [Circuit 文档](Circuit.ipynb)。

作为一个示例，终止 3 端口网络（如理想 3 路分配器）的一个端口：

<img src="figures/networks_connecting_3port_1port.svg" width="600">

可以这样做：

In [ ]:
tee = rf.data.tee

要将 tee 的端口 `1` 连接到延迟短路的端口 `0`，

In [ ]:
terminated_tee = rf.network.connect(tee, 1, delayshort, 0)
terminated_tee

`parallelconnect()` 方法也可以用稍微不同的语法处理这种情况。

In [ ]:
terminated_tee_par = rf.network.parallelconnect([tee, delayshort], [1, 0])
terminated_tee_par

在前面的示例中，3 端口网络 `tee` 的端口 #2 成为生成的 2 端口网络的端口 #1。

## 多端口网络的多次连接
多次使用 `connect` 函数时跟踪端口编号可能很繁琐（这就是为什么 [Circuit 对象](Circuit.ipynb) 可能更简单的原因）。

让我们用以下示例来说明这一点：将 tee 接头（3 端口）的端口 #1 连接到传输线（2 端口）的端口 #0：

<img src="figures/networks_connecting_3port_2port.svg" width="600">

为了跟踪连接操作后的端口方案，让我们更改端口特性阻抗（上图中红色）：

In [ ]:
tee.z0 = [1, 2, 3]
line.z0 = [10, 20]
# the resulting network is:
rf.network.connect(tee, 1, line, 0)

## 从交叉点角度连接网络
在前面的示例中，我们简要介绍了 `parallelconnect()` 方法。在本节中，我们将通过多个示例详细介绍 `parallelconnect()` 方法的应用场景，并提供三种解决方案的比较建议。

首先，让我们考虑最简单的示例：在 N 端口 `Network` 内任意连接两个端口。这将产生一个 (N-2) 端口 `Network`。

<img src="figures/networks_innerconnect_2_port.svg" width="600">

在这里，我们可以使用 `innerconnect()` 方法来连接 N 端口 `Network` 内的任意两个端口

```
# 连接 N 端口网络的第 m 和第 n 个端口
inner_network = rf.network.innerconnect(nport_network, m, n)
```

`parallelconnect()` 方法可以像这样处理内连情况：

```
inner_network_par = rf.network.parallelconnect(nport_network, [[m, n]])
```

如果你预计需要内连超过 2 个端口，`innerconnect()` 方法将不足以完成此任务。你需要考虑使用 `tee`（`splitter`）或 `Circuit` 对象来实现目标。然而，`parallelconnect()` 为此类情况提供了一个简单的解决方案，

```
# 内连 N 端口网络端口列表示例
ports_list = [[m, n, ..., y, z]]
inner_network2_par = rf.network.parallelconnect(nport_network, ports_list)
```

第二个示例涉及连接两个多端口网络。由于这种情况之前已经演示过，我们在此不再赘述。

接下来，让我们考虑多个多端口 `Networks` 的并联连接。这的一个常见应用是构建 `T 型滤波电路`，图取自 [electroniclinic.com](https://www.electroniclinic.com/filter-circuit-and-need-of-filters-in-electronics/#google_vignette)

![T 型滤波电路](figures/t_type_filter.png)

让我们忽略具体细节，仅比较在实现此示例时的三种方法，你可以使用：

```
# 1. 使用 tee 的 Connect() 方法
t_type_filter_ntwk = rf.network.connect(L1, 1, tee, 0)
t_type_filter_ntwk = rf.network.connect(t_type_filter_ntwk, 1, C, 0)
t_type_filter_ntwk = rf.network.connect(t_type_filter_ntwk, 2, L2, 0)
t_type_filter_ntwk

# 2. Circuit 对象方法
cnx = [
    [(port1, 0), (L1, 0)],
    [(port2, 0), (C , 1)],
    [(port3, 0), (L2, 1)],
    [(L1, 1), (C, 0), (L2, 0)]
]
t_type_filter_ckt = rf.cicuit.Circuit(cnx)
t_type_filter_ntwk = t_type_filter_ckt.network
t_type_filter_ntwk

# 3. Parallelconnect() 方法
t_type_filter_ntwk = rf.network.parallelconnect([L1, C, L2], [1, 0, 0])
t_type_filter_ntwk
```

可以看出，`parallelconnect()` 方法可以非常清晰简洁地实现 `T 型滤波` 电路的构建。

在本节中，我们比较了 `parallelconnect()` 方法，并将其与 `connect()/innerconnect()` 方法和 `Circuit` 对象从交叉点角度连接 `Networks` 进行了比较。

`parallelconnect()` 方法擅长以最少的努力处理多个并联连接，使其成为需要简单性和效率场景的理想选择。相比之下，`connect()/innerconnect()` 方法更适合更简单的顺序连接，而 `Circuit` 对象用于具有更大控制权的复杂多层连接。

从交叉点角度来看，`Circuit` 对象最适合管理具有多个交叉点的复杂网络，这是一项超出其他方法能力的任务，其他方法专为单一交叉点场景设计。虽然 `innerconnect()/connect()` 方法仅限于处理单个或成对网络之间的连接，`parallelconnect()` 消除了对网络数量的限制，可以在一行代码中有效地为多个网络建立连接，使其对于在多点具有并联连接的复杂电路特别有利。

## 参考文献


[1] Compton, R.C.; , "Perspectives in microwave circuit analysis," Circuits and Systems, 1989., Proceedings of the 32nd Midwest Symposium on , vol., no., pp.716-718 vol.2, 14-16 Aug 1989. URL: http://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=101955&isnumber=3167